In [ ]:
from pathlib import Path
import importlib.metadata
import json
import shutil
import subprocess
import sys
import zipfile

matches = list(Path('/kaggle/input').rglob('input_manifest.json'))
assert len(matches) == 1, f'Expected one REC-01 bundle; found {matches}'
bundle = matches[0].parent
project = Path('/kaggle/working/rec01_project')
project.mkdir(exist_ok=False)
for name in ('canonical_1000000.parquet', 'd02_manifest.json', 'input_manifest.json'):
    shutil.copy2(bundle / name, project / name)
archive = bundle / 'rec01_source.zip'
if archive.is_file():
    shutil.copy2(archive, project / archive.name)
    with zipfile.ZipFile(archive) as source_zip:
        source_zip.extractall(project)
else:
    source_tree = bundle / 'rec01_source'
    if not (source_tree / 'src').is_dir():
        source_tree = bundle
    for name in ('src', 'scripts'):
        shutil.copytree(source_tree / name, project / name)

try:
    duckdb_version = importlib.metadata.version('duckdb')
except importlib.metadata.PackageNotFoundError:
    duckdb_version = None
if duckdb_version != '1.5.6':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb==1.5.6'], check=True)

import torch
assert torch.cuda.is_available(), 'GPU is not enabled in this Kaggle notebook'
print(json.dumps({'bundle': str(bundle), 'python': sys.version,
                  'torch': torch.__version__, 'cuda': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(0),
                  'duckdb': importlib.metadata.version('duckdb')}, sort_keys=True), flush=True)

subprocess.run([sys.executable, '-u', '-m', 'scripts.rec01_kaggle_prep',
                'verify-input', '--bundle', str(project)], cwd=project, check=True)
run = project / 'REC01-Kaggle-1M'
subprocess.run([sys.executable, '-u', '-m', 'src.models.rec01_train',
                '--source', str(project / 'canonical_1000000.parquet'),
                '--out', str(run), '--device', 'cuda'], cwd=project, check=True)
subprocess.run([sys.executable, '-u', '-m', 'scripts.rec01_kaggle_prep',
                'seal-output', '--bundle', str(project), '--run', str(run)],
               cwd=project, check=True)
print('REC01_KAGGLE_PASS', run, flush=True)
